# Day 25 — Dashboard Foundation

Run from the repository root in the `ipl-player-performance` environment. This creates a separate Bootstrap dashboard template, CSS and JavaScript files without overwriting the existing dashboard. It does not assume API routes are already integrated.

In [1]:
from pathlib import Path
import subprocess, sys, shutil

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "django_api").is_dir():
    roots = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "django_api").is_dir()]
    if roots: REPO_ROOT = roots[0]
DJANGO_ROOT = REPO_ROOT / "django_api"
APP_DIR = DJANGO_ROOT / "ipl_api"
MANAGE_PY = DJANGO_ROOT / "manage.py"
assert MANAGE_PY.exists(), f"Missing {MANAGE_PY}"
TEMPLATES = APP_DIR / "templates" / "ipl_api"
CSS_DIR = APP_DIR / "static" / "ipl_api" / "css"
JS_DIR = APP_DIR / "static" / "ipl_api" / "js"
print("Repository:", REPO_ROOT)
print("App:", APP_DIR)
for p in [APP_DIR / "urls.py", APP_DIR / "views.py", TEMPLATES, CSS_DIR, JS_DIR]:
    print(("EXISTS " if p.exists() else "MISSING "), p)


Repository: D:\Git\Personal\ipl-player-performance
App: D:\Git\Personal\ipl-player-performance\django_api\ipl_api
EXISTS  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
EXISTS  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\views.py
MISSING  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api
MISSING  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\css
MISSING  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\js


In [2]:
TEMPLATES.mkdir(parents=True, exist_ok=True)
CSS_DIR.mkdir(parents=True, exist_ok=True)
JS_DIR.mkdir(parents=True, exist_ok=True)

template_path = TEMPLATES / "dashboard_day25.html"
css_path = CSS_DIR / "dashboard_day25.css"
js_path = JS_DIR / "dashboard_day25.js"

template = '''{% load static %}
<!doctype html><html lang="en"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>IPL Player Analytics</title>
<link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
<link rel="stylesheet" href="{% static 'ipl_api/css/dashboard_day25.css' %}">
</head><body>
<nav class="navbar navbar-dark bg-dark px-3"><span class="navbar-brand">IPL Player Analytics</span><span class="text-light small">Performance • EDA • Predictions</span></nav>
<main class="container-fluid py-4">
<div class="d-flex flex-wrap justify-content-between align-items-center gap-2 mb-4"><div><h1 class="h3 mb-1">Player Performance Dashboard</h1><p class="text-secondary mb-0">Explore historical form and model predictions.</p></div><button id="refreshButton" class="btn btn-primary">Refresh summary</button></div>
<div id="statusMessage" class="alert alert-secondary" role="status">Dashboard shell loaded. API connection has not yet been verified.</div>
<div class="row g-3 mb-4">
<div class="col-12 col-md-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Players</div><div class="metric-value" id="playersMetric">—</div></div></div></div>
<div class="col-12 col-md-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Matches</div><div class="metric-value" id="matchesMetric">—</div></div></div></div>
<div class="col-12 col-md-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Batting records</div><div class="metric-value" id="battingMetric">—</div></div></div></div>
<div class="col-12 col-md-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Bowling records</div><div class="metric-value" id="bowlingMetric">—</div></div></div></div>
</div>
<div class="row g-3"><div class="col-12 col-lg-6"><div class="card h-100"><div class="card-header">API connection</div><div class="card-body">Summary endpoint: <code>/api/eda/summary/</code><p class="small text-secondary mt-3">Values populate only for recognized fields in the actual API response.</p></div></div></div>
<div class="col-12 col-lg-6"><div class="card h-100"><div class="card-header">Integration checklist</div><div class="card-body"><ul><li>Register a dashboard-render URL.</li><li>Inspect the actual summary JSON response.</li><li>Map response fields and then add player search/charts.</li></ul></div></div></div></div>
</main><script src="{% static 'ipl_api/js/dashboard_day25.js' %}"></script></body></html>'''
css = '''body { background: #f5f7fb; color: #172033; }
.metric-card { border: 0; box-shadow: 0 2px 12px rgba(25,35,60,.06); min-height: 118px; }
.metric-value { font-size: 2rem; font-weight: 700; margin-top: .25rem; }
.card { border-color: #e7eaf1; }
'''
js = '''(() => {
  const status = document.getElementById("statusMessage");
  document.getElementById("refreshButton").addEventListener("click", async () => {
    status.className = "alert alert-info";
    status.textContent = "Checking /api/eda/summary/…";
    try {
      const response = await fetch("/api/eda/summary/", {headers: {"Accept": "application/json"}});
      if (!response.ok) throw new Error(`HTTP ${response.status}`);
      const data = await response.json();
      const aliases = {
        playersMetric: ["players", "player_count", "total_players"],
        matchesMetric: ["matches", "match_count", "total_matches"],
        battingMetric: ["batting_records", "batting_count"],
        bowlingMetric: ["bowling_records", "bowling_count"]
      };
      for (const [id, keys] of Object.entries(aliases)) {
        const key = keys.find(k => Object.prototype.hasOwnProperty.call(data, k));
        if (key) document.getElementById(id).textContent = data[key];
      }
      status.className = "alert alert-success";
      status.textContent = "Summary endpoint responded. Verify metric field mappings against the actual response.";
    } catch (error) {
      status.className = "alert alert-warning";
      status.textContent = `Could not load summary: ${error.message}. Check route and server.`;
    }
  });
})();'''
for path, content in [(template_path, template), (css_path, css), (js_path, js)]:
    if path.exists():
        backup = path.with_suffix(path.suffix + ".day25_backup")
        shutil.copy2(path, backup)
        print("Backed up:", backup)
    path.write_text(content, encoding="utf-8")
    print("Created:", path)


Created: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api\dashboard_day25.html
Created: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\css\dashboard_day25.css
Created: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\js\dashboard_day25.js


In [3]:
p = subprocess.run([sys.executable, str(MANAGE_PY), "check"], cwd=str(DJANGO_ROOT), text=True, capture_output=True)
print("Django check return code:", p.returncode)
print(p.stdout)
if p.stderr: print(p.stderr)


Django check return code: 0
System check identified no issues (0 silenced).



## Required manual integration

1. Add a view that renders `ipl_api/dashboard_day25.html`.
2. Merge a dashboard URL into the existing `django_api/ipl_api/urls.py`; do not replace existing routes.
3. Start the server and open the dashboard route.
4. Inspect the real `/api/eda/summary/` JSON before adjusting the metric aliases.
5. Add charts and prediction forms only after the response shapes are confirmed.

The notebook creates a dashboard shell, not a fully integrated dashboard. Browser/API testing is still required.